In [ ]:
import torch
from unsloth import FastLanguageModel
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from pathlib import Path

persist_directory = "/workspaces/Arch_PC_Assistent/embed_model"
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
vectorstore = Chroma(embedding_function=embeddings,
                     persist_directory=persist_directory)


path_model_v1 = "/workspaces/Arch_PC_Assistent/outputs/SFT/arch_assistant_lora_1"
max_seq_length = 4096 
model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit" 

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = path_model_v1,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

In [ ]:
import json
import os
import time
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv

# 1. Setup
load_dotenv()
client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1"
)

rohdaten_datei = "/workspaces/Arch_PC_Assistent/dataset/raw_generations.jsonl" 
gold_datei = "/workspaces/Arch_PC_Assistent/dataset/rsft_gold_dataset.jsonl"

bereits_verarbeitet = set()
if os.path.exists(gold_datei):
    with open(gold_datei, "r", encoding="utf-8") as f:
        for line in f:
            try:
                data = json.loads(line)
                bereits_verarbeitet.add(data["instruction"]) 
            except json.JSONDecodeError:
                continue

zu_verarbeitende_samples = []

with open(rohdaten_datei, "r", encoding="utf-8") as f:
    for line in f:
        try:
            sample = json.loads(line)
            # Only add if the question is not already in the gold dataset
            if sample["instruction"] not in bereits_verarbeitet:
                zu_verarbeitende_samples.append(sample)
        except json.JSONDecodeError:
            continue

print(f"Already in gold dataset: {len(bereits_verarbeitet)}")
print(f"Starting API evaluation for the remaining {len(zu_verarbeitende_samples)} questions...")

# ==========================================
# 4. THE JUDGE FUNCTION
# ==========================================
def evaluate_single_answer(question, rag_context, answer):
    """Evaluates a SINGLE answer. Returns the score dictionary or None on error."""
    judge_prompt = f"""You are an expert AI judge evaluating answers for an advanced Arch Linux, Hyprland, and Zsh Assistant.
QUESTION: {question}

PROVIDED RAG CONTEXT (Source of Truth):
{rag_context}

MODEL ANSWER TO EVALUATE:
{answer}

Evaluate strictly on two criteria:
1. Format: Did the model use EXACTLY `<think>` tags for reasoning and `<answer>` tags for the final solution? (Score 0 or 1)
2. Accuracy: Is the answer technically correct based on the context (or general Arch Linux, Hyprland, and Zsh knowledge if the context was incomplete) and free of hallucinations? (Score 1-5, where 4 is good and 5 is perfect).

Return ONLY a valid JSON object in this exact format:
{{"format_score": 1, "accuracy_score": 5, "reason": "brief explanation"}}"""

    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": judge_prompt}],
            temperature=0.0, 
            response_format={"type": "json_object"}
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"\n[API Error] {e}")
        time.sleep(2) 
        return None

with open(gold_datei, "a", encoding="utf-8") as outfile:
    
    for sample in tqdm(zu_verarbeitende_samples, desc="Evaluating answers"):
        
        # Extract everything directly from the current JSON object
        frage = sample["instruction"]
        kontext = sample["input"]
        generierte_antworten = sample["generated_outputs"] 
        
        perfekte_antwort_gefunden = False
        
        # Iterate through the up to 6 answers
        for antwort in generierte_antworten:
            
            # Pre-filter: Saves API costs if the required format tags are missing anyway
            if "<think>" not in antwort or "<answer>" not in antwort:
                continue 
                
            bewertung = evaluate_single_answer(frage, kontext, antwort)
            
            if bewertung:
                if bewertung.get("format_score") == 1 and bewertung.get("accuracy_score") >= 4:
                    
                    # SUCCESS! Build the final gold format for Unsloth
                    gold_sample = {
                        "instruction": frage,
                        "input": kontext,
                        "output": antwort
                    }
                    
                    # Write to disk immediately
                    outfile.write(json.dumps(gold_sample, ensure_ascii=False) + "\n")
                    outfile.flush()
                    
                    perfekte_antwort_gefunden = True
                    break # EARLY EXIT! Breaks the loop and saves API calls for the remaining answers.

print("\nRSFT filtering completed! Your training dataset is ready!")